In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [47]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
# model = ChatGoogleGenerativeAI(model = "gemini-3-flash-preview")
model = ChatOpenAI(model = "gpt-5-nano")

In [12]:
# Used to send one request
response = model.invoke("what is CNN. answer will be very short")
print(response.content)

CNN can mean: 
- Cable News Network (24/7 US news channel), or 
- Convolutional Neural Network (a type of deep learning model). 

Which one did you mean?


In [14]:
# Multiple prompts together.
question = [
    "What is machine learning",
    "what is CNN",
    "what is Deep learning"
]

response = model.batch(question)

for i in response:
    print(i.content)

Machine learning is a field of artificial intelligence that focuses on building systems that learn from data to make predictions or decisions, rather than being explicitly programmed for every task.

Key ideas:
- Instead of giving a computer explicit rules, you provide data and a goal, and the model learns patterns from the data to achieve that goal.
- A typical workflow: collect data, preprocess it, choose a model, train it by optimizing a loss function, evaluate how well it generalizes to new data, and deploy it.

Main types:
- Supervised learning: learn from labeled examples (inputs with known outputs), e.g., predicting house prices, spam detection.
- Unsupervised learning: find structure in unlabeled data, e.g., clustering customers, reducing dimensions with PCA.
- Reinforcement learning: an agent learns by interacting with an environment to maximize rewards, e.g., playing a game or controlling a robot.

How it works in simple terms:
- You have data (features) and a target.
- The m

In [24]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()
prompt = ChatPromptTemplate.from_messages(
   [
       ("system","Explain {topic} in simple term")
   ]
)

chain = prompt | model | output_parser

chain.invoke({"topic":"neural network"})

'Here’s a simple way to think about a neural network:\n\n- It’s a computer program that tries to learn from examples, in a way that mimics how brain neurons work, but with simple math instead of real biology.\n\n- Structure: it’s made of many tiny units called neurons arranged in layers. There’s:\n  - an input layer (reads the data you give it),\n  - one or more hidden layers (do the thinking),\n  - an output layer (gives the answer).\n\n- How it works (in plain terms):\n  - Each neuron takes some numbers from the previous layer, multiplies them by small numbers called weights, adds a little bias, and then squashes the result with a simple rule (an activation function).\n  - The result is passed to the next layer, and so on, until you get a final result from the output layer.\n\n- How it learns:\n  - You show it many examples where you know the right answer (e.g., pictures labeled “cat” or “dog”).\n  - It makes a prediction, compares it to the real answer, and measures how wrong it was

In [26]:
# streaming

for chunk in model.stream("Explain Machine learning"):
    print(chunk.content, end = "")

Machine learning is a branch of artificial intelligence where computer systems learn from data to perform tasks without being explicitly programmed for every situation. Instead of writing rules by hand, you give a model examples and let it figure out patterns.

Key ideas
- Data, model, and objective: You provide data (inputs and usually outputs), choose a model (a mathematical function with adjustable parameters), and define a goal to optimize (loss or cost) that measures how wrong the model is.
- Learning from experience: The model’s parameters are adjusted to minimize the loss on the training data, so it can make accurate predictions on new, unseen data.

Common types of learning
- Supervised learning: The most common type. You train on labeled data (inputs with known outputs). Tasks include:
  - Regression: predicting a continuous value (e.g., house prices).
  - Classification: predicting a discrete label (e.g., spam vs. not spam).
- Unsupervised learning: No labels. You discover st

In [27]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="you will explain user question in a short single sentence"),
    HumanMessage(content= "What is agentic ai?")
]

response = model.invoke(messages)
response.content

'Agentic AI is artificial intelligence designed to act autonomously and purposefully—perceiving its environment, setting goals, planning, and taking actions without needing explicit step-by-step human instructions.'

In [33]:
# structured output using pydantic

from pydantic import BaseModel
from langchain_core.output_parsers import StrOutputParser
from langchain.agents import create_agent

class Student_info(BaseModel):
    name : str
    id: str
    year: int

agent = create_agent(model, response_format= Student_info)
result = agent.invoke({
    "messages":[{"role":"user","content":"My name is rachin. my id is 221-15-5466, i started my journey in 2026"}]

})

print(result["structured_response"])


name='rachin' id='221-15-5466' year=2026


In [ ]:

# structured output with pydantic

from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import List, Optional

# Define the output structure
class MovieReview(BaseModel):
    title: str = Field(description="Movie title")
    rating: float = Field(description="Rating from 0 to 10")
    pros: List[str] = Field(description="Good points about the movie")
    cons: List[str] = Field(description="Bad points about the movie")

    recommended: bool = Field(description="Should people watch it?")
    genre: Optional[str] = None

# Force the LLM to return a MovieReview
agent = create_agent(model, response_format=MovieReview)

review = agent.invoke(
    {
  
        "messages":[{"role":"user","content":"review the movie inception (2010)"}]
    }
)

print(review['structured_response'])

title='Inception' rating=9.0 pros=['Inventive, high-concept premise of shared dreaming and multi-layered heists', 'Spectacular visual effects and practical stunt work', "Memorable score by Hans Zimmer, especially the ticking motif in 'Time'", 'Strong ensemble performances (DiCaprio, Gordon-Levitt, Hardy, Page)', "Smart, emotional core driving the plot (Cobb's guilt and longing)", 'Clever, tightly constructed screenplay with rewarding rewatch value', "Nolan's direction creates a cohesive logic within a dream-in-dream framework"] cons=['Complex plot that can be hard to follow on first viewing', 'Some characters underdeveloped or sidelined', 'Some exposition-heavy stretches that slow momentum', 'Ambiguous ending may frustrate viewers seeking a definitive conclusion'] recommended=True genre='Science fiction, psychological thriller'


In [ ]:

# typedDict is a python build in way to type-hint dictionaries. it is faster than pydantic. use this when validation is not needed

from langchain_openai import ChatOpenAI
from typing_extensions import TypedDict, Annotated
from typing import List

# Define output as TypedDict
class ProductInfo(TypedDict):
    name: Annotated[str, "Product name"]
    price: Annotated[float, "Price in USD"]
    features: Annotated[List[str], "List of key features"]
    in_stock: Annotated[bool, "Whether item is available"]
    category: Annotated[str, "Product category"]

llm = ChatOpenAI(model="gpt-4o-mini")
structured_llm = llm.with_structured_output(ProductInfo)

product = structured_llm.invoke(
    "Describe the Apple MacBook Pro M3"
)

print(type(product))        # &lt;class 'dict'&gt; — not a Pydantic object!
print(product["name"])     # "MacBook Pro M3"
print(product["price"])    # 1999.0
print(product["features"]) # ["Apple M3 chip", ...]

# Pydantic → object access: product.name
# TypedDict → dict access: product["name"]

<class 'dict'>
Apple MacBook Pro M3
1999.99
['Apple M3 chip', 'Liquid Retina XDR display', '16-core Neural Engine', 'Up to 18 hours battery life', 'ProMotion technology with adaptive refresh rates', 'macOS Monterey', 'Touch Bar', 'Thunderbolt 4 ports', 'High-fidelity six-speaker sound system']


In [54]:

from langchain_openai import ChatOpenAI
from dataclasses import dataclass, field
from typing import List

# Define output as a dataclass
@dataclass
class RecipeOutput:
    name: str
    ingredients: List[str]
    steps: List[str]
    prep_time_minutes: int
    difficulty: str  # "easy", "medium", "hard"

llm = ChatOpenAI(model="gpt-4o-mini")
structured_llm = llm.with_structured_output(RecipeOutput)

recipe = structured_llm.invoke(
    "Give me a recipe for chicken biryani"
)

# # Object attribute access (like Pydantic, unlike TypedDict)
# print(recipe.name)               # "Chicken Biryani"
# print(recipe.prep_time_minutes)  # 45
# print(recipe.difficulty)         # "medium"

# # Dataclass auto-generates __repr__
# print(recipe)
# # RecipeOutput(name='Chicken Biryani', ...)

# # Compare all three approaches:
# # TypedDict  → result["name"]      (dict)
# # Dataclass  → result.name         (object, no validation)
# # Pydantic   → result.name         (object, WITH validation)

print(recipe['name'])

Chicken Biryani


In [59]:
## summarization : Summarization is one of the most common LangChain use cases. The challenge: LLMs have a limited context window. A 100-page document won't fit. So we need to summarize the content

from langchain_classic.chains import load_summarize_chain
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 100)

long_text = ''' Machine learning (ML) is one of the most transformative technologies of the modern era, enabling computers to learn patterns, make decisions, and improve performance without being explicitly programmed for every task. At its core, machine learning is a branch of Artificial Intelligence that focuses on building algorithms capable of analyzing data and making predictions or decisions based on that data. Instead of following rigid instructions written by humans, ML systems learn from examples, much like humans learn from experience. For example, when a machine learning model is trained with thousands of images of cats and dogs, it gradually learns the patterns that distinguish one animal from another. This ability to learn from data has made machine learning extremely valuable in many industries, including healthcare, finance, transportation, agriculture, education, cybersecurity, and entertainment. In healthcare, ML models help doctors detect diseases such as cancer from medical images with remarkable accuracy. In finance, banks use machine learning to identify fraudulent transactions and predict market trends. Social media platforms rely on ML algorithms to recommend content, while streaming services suggest movies and music based on user preferences. One of the major reasons behind the popularity of machine learning is the rapid growth of data and computing power. Every day, enormous amounts of data are generated from smartphones, sensors, websites, cameras, and online transactions. Machine learning algorithms use this data to discover hidden insights that would be difficult or impossible for humans to identify manually. There are several major types of machine learning, including supervised learning, unsupervised learning, and reinforcement learning. In supervised learning, models are trained using labeled data, meaning the correct answers are already known during training. Common examples include spam email detection and house price prediction. In unsupervised learning, the model tries to find patterns or groupings in unlabeled data, such as customer segmentation in marketing. Reinforcement learning involves training an agent through rewards and penalties, allowing it to learn optimal actions over time; this approach is widely used in robotics and game-playing AI systems. Deep learning, a specialized subset of machine learning inspired by the structure of the human brain, has recently achieved groundbreaking success in areas such as computer vision, natural language processing, and speech recognition. Technologies like self-driving cars, virtual assistants, and advanced chatbots are powered by deep learning models. Despite its incredible capabilities, machine learning also presents challenges and ethical concerns. Issues such as biased data, lack of transparency, privacy risks, and high computational costs must be carefully managed to ensure responsible use of AI technologies. Researchers worldwide are actively working on improving model fairness, interpretability, and energy efficiency. As technology continues to evolve, machine learning is expected to play an even greater role in shaping the future of society. From scientific discoveries to personalized education and smart healthcare systems, ML has the potential to improve human life in countless ways. Understanding machine learning is becoming increasingly important not only for researchers and engineers but also for students, business leaders, and policymakers, as its influence continues to expand across nearly every aspect of modern life.
'''


docs = text_splitter.create_documents([long_text])


chain_refine = load_summarize_chain(model, chain_type="map_reduce") #refine can be use but it is very slow

summary = chain_refine.invoke(docs)

print(summary["output_text"])


Machine learning (ML) is a branch of AI that uses data-driven algorithms to learn patterns from data, enabling predictions and decisions without explicit programming. It is applied across sectors such as healthcare, finance, transportation, agriculture, education, cybersecurity, and entertainment, with tasks like disease detection, fraud detection, market prediction, and personalized recommendations. ML comprises supervised, unsupervised, and reinforcement learning, with deep learning driving major advances in vision, NLP, and speech (e.g., self-driving cars, virtual assistants). It also raises concerns about bias, transparency, privacy, and computational costs, guiding efforts in responsible AI to improve privacy, fairness, interpretability, and energy efficiency as ML continues to shape science and society.
